# 11. 보드와 이야기하기 — 시리얼(pyserial)

> USB 로 꽂은 보드(ESP32 등)와 **글자 한 줄씩** 주고받는다. AI 결과로 LED · 모터를 움직이거나, 센서 값을 받아 기록할 때 쓴다.

### 이 시간에 배우는 것
1. 시리얼 통신 기초: 포트 · 속도(baud) · 줄 끝(EOL)
2. pyserial 로 열기 · 보내기 · 받기 · 닫기
3. 간단한 약속(프로토콜) 만들기: 되돌림(echo) 무시, 머리말(prefix)로 답 고르기
4. 받은 줄에서 숫자 꺼내기

### 먼저 — 안전
- edge-lab 의 [보드] 앱이 포트를 쓰고 있으면 여기서 못 연다 — 보드 앱에서 연결을 끊거나 edge-lab 을 끈다.
- 보드는 USB 로만 연결한다. **서보 · 모터처럼 전류가 큰 부품의 전원은 USB 에서 끌어오지 말고 따로** 준다
  (여러 개가 한꺼번에 움직이면 전압이 떨어져 보드가 다시 켜진다). 외부 전원과 보드는 GND 를 같이 묶는다.
- ESP32 의 GPIO 는 **3.3V** 다. 5V 센서 출력을 바로 잇지 않는다 (전압 분배 · 레벨 변환).

### 용어
| 용어 | 뜻 |
|---|---|
| 포트 | 윈도우 `COM3`, 우분투 `/dev/ttyUSB0` · `/dev/ttyACM0` |
| baud | 1초에 보내는 비트 수. **양쪽이 같아야** 한다 (보통 115200) |
| 줄 끝(EOL) | 한 줄이 끝났다는 표시. `\n` 또는 `\r\n` — 펌웨어와 맞춘다 |
| DTR / RTS | 제어선. 많은 USB-UART 보드가 이 선으로 **리셋**된다 |

## 1. 포트 찾기

In [ ]:
import os
import re
import time
import serial
import serial.tools.list_ports

ports = list(serial.tools.list_ports.comports())
for p in ports:
    print(f"{p.device:14s} {p.description}  (VID {p.vid:04X})" if p.vid else f"{p.device:14s} {p.description}")
PORT = os.environ.get("BOARD_PORT") or (ports[0].device if ports else None)
print("쓸 포트:", PORT, "← 다르면 직접 적는다: PORT = 'COM5'")

VID 로 칩을 알 수 있다: `303A` Espressif(ESP32 내장 USB) · `10C4` CP210x · `1A86` CH340 · `0403` FTDI · `2E8A` RP2040.

## 2. 열기
DTR/RTS 를 **내린 채** 열면 여는 순간 보드가 리셋되는 것을 막는다(CP210x · CH340 보드). ESP32-S3 내장 USB 에서도 막히는지는 보드마다 다르다 — 확인 필요.
`timeout=0.2` — `readline()` 이 줄을 기다리는 최대 시간(초). 이게 없으면 줄이 안 올 때 영원히 기다린다.

In [ ]:
BAUD = 115200
EOL = "\n"                                   # 줄 끝 — 펌웨어가 \r\n 을 기다리면 바꾼다

ser = serial.Serial()
ser.port, ser.baudrate, ser.timeout = PORT, BAUD, 0.2
ser.dtr = ser.rts = False
ser.open()
time.sleep(0.3)
ser.reset_input_buffer()                     # 열기 전에 쌓인 줄은 버린다
print("열림:", ser.name)

## 3. 보내기 · 보내고 답 받기
- 글은 **바이트**로 보낸다: `"PING\n".encode("utf-8")`
- 받은 바이트는 글로 되돌린다: `.decode("utf-8", errors="replace")` → 깨진 글자가 있어도 멈추지 않게
- 보드가 명령을 그대로 되돌려 보내는(echo) 경우가 있다 — 보낸 글과 같은 줄은 답으로 치지 않는다
- 보드가 센서 값을 계속 흘려보내는 중이면 그 줄이 답으로 잡힐 수 있다 → **머리말(prefix)** 로 고른다

In [ ]:
def send(text):
    ser.write((text + EOL).encode("utf-8"))

def read_line():
    return ser.readline().decode("utf-8", errors="replace").strip()

def ask(text, wait=1.0, prefix=""):
    """보내고, 그 뒤 처음 들어온 (prefix 로 시작하는) 줄을 돌려준다. 없으면 None."""
    ser.reset_input_buffer()
    send(text)
    end = time.time() + wait
    while time.time() < end:
        line = read_line()
        if line and line != text and line.startswith(prefix):
            return line
    return None

print("답:", ask("PING"))
print("답(PO 로 시작하는 것만):", ask("PING", prefix="PO"))

## 4. 보드가 스스로 보내는 줄 읽기 — 숫자 꺼내기
`T=23.5` 같은 줄에서 숫자만 꺼낼 때 정규식이 편하다: `re.search(r"T=([-\d.]+)", line)`.

In [ ]:
def number_of(line, key):
    m = re.search(rf"{key}=([-\d.]+)", line)
    return float(m.group(1)) if m else None

end = time.time() + 5
while time.time() < end:
    line = read_line()
    if line:
        print(f"{time.strftime('%H:%M:%S')}  {line:20s} → T 값: {number_of(line, 'T')}")

## 자주 나는 오류
| 증상 | 원인 · 고침 |
|---|---|
| `PermissionError` / `could not open port` | 다른 프로그램(보드 앱 · 아두이노 IDE 시리얼 모니터)이 쓰는 중. 우분투는 `dialout` 그룹 |
| 글자가 깨짐 | baud 가 다르다 |
| 답이 안 옴 | 줄 끝(EOL)이 펌웨어와 다르다 · 펌웨어가 그 명령을 모른다 |
| 열 때마다 보드가 다시 켜짐 | DTR/RTS 리셋 → 내린 채 연다. 그래도 되면 보드 회로 문제 — 확인 필요 |

## 정리
- 시리얼 = 포트 · baud · 줄 끝을 양쪽이 맞추고, 글 한 줄씩 바이트로 주고받기.
- 답 고르기: 되돌림(echo) 무시 + 머리말(prefix). 기다림에는 꼭 시간 제한.
- 다 쓰면 닫는다 — 다른 프로그램이 포트를 쓸 수 있게.

---
# 🛠 미니 프로젝트 — 센서 기록기 + 원격 LED

**만들 것**: 20초 동안 보드가 보내는 센서 값(`T=…`)을 모아 **CSV 파일**로 저장하고 **그래프**로 그린다.
값이 기준(`LIMIT`)을 넘으면 `LED ON`, 내려가면 `LED OFF` 를 보낸다 (바뀔 때만 보낸다).

**쓰는 것**: `read_line` · `number_of` · `send` (이 노트북), `csv`, `matplotlib`

명령 이름(`LED ON` 등)은 펌웨어가 정한다 — 내 펌웨어에 맞게 바꾼다.

In [ ]:
import csv
import matplotlib.pyplot as plt     # 그래프 글자는 영어로

LIMIT = 25.0
SECONDS = 20
rows, led = [], None
t0 = time.time()
while time.time() - t0 < SECONDS:
    line = read_line()
    v = number_of(line, "T") if line else None
    if v is None:
        continue
    rows.append((round(time.time() - t0, 2), v))
    want = v > LIMIT
    if want != led:                                   # 바뀔 때만 보낸다
        send("LED ON" if want else "LED OFF")
        led = want
        print(f"{rows[-1][0]:5.1f}s  T={v} → {'LED ON' if want else 'LED OFF'}")

with open("sensor_log.csv", "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerows([("sec", "T")] + rows)
print(f"{len(rows)}개 저장: sensor_log.csv")

if rows:
    xs, ys = zip(*rows)
    plt.figure(figsize=(7, 2.5)); plt.plot(xs, ys, marker=".")
    plt.axhline(LIMIT, color="r", ls="--"); plt.xlabel("seconds"); plt.ylabel("T"); plt.show()

In [ ]:
ser.close()                                   # 다 쓰면 꼭 닫는다
print("닫힘")

**더 해 보기**
- 06 번의 손가락 수를 보드로 보내 LED 개수로 켜기 (`send(f"LED {n}")`)
- 기준값을 한 번 넘었다고 바로 켜지 말고 3번 연속 넘을 때만 켜기 (흔들림 막기)
- CSV 를 엑셀에서 열어 보기